Nous étudions la méthode ARBRE de DECISION pour régression

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt

In [ ]:
# Générer les nombres aléatoires
generateur = np.random.RandomState(1)
# Tableau trié contenant 200 nombres générés aléatoirement
x = np.sort(5 * generateur.rand(200, 1), axis=0)

# Aplatir le tableau en utilisant ravel
y = np.sin(x).ravel()

# Ajout d'un bruit
y += 0.2 * (0.5 - generateur.rand(200))

In [ ]:
# Les données générées
plt.scatter(x, y, c="red", label="données")
plt.xlabel("données")
plt.ylabel("cible")

In [ ]:
# Définir le modèle
model_DTReg = DecisionTreeRegressor(max_depth=3)
# Apprentissage
model_DTReg.fit(x, y)

# Générer les données de test
x_test = np.arange(0.0, 5.0, 0.01)
# Transformation d'un tableau 1D en un tableau N*1
x_test = x_test[:, np.newaxis]

prediction = model_DTReg.predict(x_test)

# Affichage des résultats et les données d'apprentissage
plt.figure()
plt.scatter(x, y, c="red", label="données")
plt.plot(x_test, prediction, color="blue", label="profondeurArbre = 3",
         linewidth=2)
plt.xlabel("données")
plt.ylabel("cible")
plt.title("Arbre de régression")
plt.legend()
plt.show()

In [ ]:
from sklearn import metrics

# ATTENTION : r2 et MSE doivent comparer y_test et prediction (pas x_test !)
print(" r2 = {} ".format(metrics.r2_score(y, model_DTReg.predict(x))))
print("MSE = {} ".format(metrics.mean_squared_error(y, model_DTReg.predict(x))))

In [ ]:
from sklearn.tree import export_graphviz

# Export the decision tree to a tree.dot file
# for visualizing the plot easily anywhere
export_graphviz(model_DTReg, out_file='tree.dot')

Afficher le graphe du fichier tree.dot

http://www.webgraphviz.com/

**Analyse du graphe :**

- Avec `max_depth=3`, l'arbre a au maximum 2^3 = 8 feuilles, donc **8 étiquettes (valeurs) possibles** en sortie.
- La qualité n'est pas définie par le paramètre `gini` (qui est pour la classification) mais par **`squared_error`** (erreur quadratique moyenne) pour la régression.

**Prédictions pour des valeurs spécifiques :**

In [ ]:
# Prédictions pour x = 1, 2.5, 3.5, 4
valeurs = np.array([[1], [2.5], [3.5], [4]])
predictions = model_DTReg.predict(valeurs)
for v, p in zip(valeurs.ravel(), predictions):
    print(f"x = {v} => prédiction = {p:.4f}")

**Répéter pour profondeur = 4**

On attend plus de feuilles (2^4 = 16 max) donc plus de valeurs distinctes, et une courbe plus « en escalier » qui colle mieux aux données mais risque de sur-apprendre.

In [ ]:
model_depth4 = DecisionTreeRegressor(max_depth=4)
model_depth4.fit(x, y)

prediction4 = model_depth4.predict(x_test)

plt.figure()
plt.scatter(x, y, c="red", label="données")
plt.plot(x_test, prediction, color="blue", label="profondeur = 3", linewidth=2)
plt.plot(x_test, prediction4, color="green", label="profondeur = 4", linewidth=2)
plt.xlabel("données")
plt.ylabel("cible")
plt.title("Arbre de régression : profondeur 3 vs 4")
plt.legend()
plt.show()

print("Profondeur 3 - R² :", metrics.r2_score(y, model_DTReg.predict(x)))
print("Profondeur 4 - R² :", metrics.r2_score(y, model_depth4.predict(x)))

# Prédictions pour les mêmes valeurs
for v, p3, p4 in zip(valeurs.ravel(), model_DTReg.predict(valeurs), model_depth4.predict(valeurs)):
    print(f"x = {v} | depth=3 => {p3:.4f} | depth=4 => {p4:.4f}")

**Application au dataset Diabetes**

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

data = load_diabetes()
X_all = data.data
y_all = data.target

# On utilise uniquement l'attribut IMC (index 2), comme dans le TP3
X_bmi = X_all[:-50, 2].reshape(-1, 1)
X_bmi_test = X_all[-50:, 2].reshape(-1, 1)
y_train_d = y_all[:-50]
y_test_d = y_all[-50:]

# Arbres de décision avec différentes profondeurs
for depth in [2, 3, 4, 5]:
    dt = DecisionTreeRegressor(max_depth=depth)
    dt.fit(X_bmi, y_train_d)
    y_pred_d = dt.predict(X_bmi_test)
    mse_d = metrics.mean_squared_error(y_test_d, y_pred_d)
    r2_d = metrics.r2_score(y_test_d, y_pred_d)
    print(f"Profondeur {depth} => MSE : {mse_d:.2f} | R² : {r2_d:.4f}")

# Visualisation pour depth=3
dt3 = DecisionTreeRegressor(max_depth=3)
dt3.fit(X_bmi, y_train_d)

x_range = np.linspace(X_bmi.min(), X_bmi.max(), 300).reshape(-1, 1)
plt.figure(figsize=(10, 6))
plt.scatter(X_bmi, y_train_d, c='red', s=10, label='train')
plt.scatter(X_bmi_test, y_test_d, c='green', s=20, label='test (réel)')
plt.plot(x_range, dt3.predict(x_range), color='blue', linewidth=2, label='Arbre depth=3')
plt.xlabel('IMC (standardisé)')
plt.ylabel('Progression du diabète')
plt.title('Arbre de décision (régression) sur dataset Diabetes')
plt.legend()
plt.grid(True)
plt.show()